# Dashboard Interativo – Eficiência Energética e Mobilidade Elétrica

Este dashboard apresenta uma visualização interativa dos principais indicadores associados à modernização da iluminação pública e à viabilidade de integração de carregadores para veículos elétricos (VE).

A análise inclui:

- perfis horários de consumo antes e depois da modernização LED;
- capacidade instalada e disponível nos PTD;
- estimativa da potência libertada;
- cenários de integração de carregadores VE;
- mapa ilustrativo das zonas analisadas.

In [1]:
import os
from pathlib import Path

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import ipywidgets as widgets
from IPython.display import display, Markdown

# Garantir que o notebook encontra a pasta do projeto
notebook_dir = Path.cwd()
if notebook_dir.name == "src":
    os.chdir(notebook_dir.parent)

# Caminho do dataset
caminho_dataset = Path("data/dataset_final.csv")

if not caminho_dataset.exists():
    raise FileNotFoundError(f"Ficheiro não encontrado: {caminho_dataset.resolve()}")

# Carregar dataset final consolidado
df_dashboard = pd.read_csv(caminho_dataset)

# Criar coluna categórica de viabilidade, caso não exista
df_dashboard["Viabilidade_VE"] = np.where(
    df_dashboard["D"] >= 0,
    "Viável",
    "Não Viável"
)

print("Dataset carregado com sucesso.")
print(f"Número de concelhos: {len(df_dashboard)}")
display(df_dashboard.head())

Dataset carregado com sucesso.
Número de concelhos: 278


,Distrito,Concelho,CodDistritoConcelho,P_IP_TOTAL,P_IP_Inef,Rate_Ineficiencia,Cap_PTD,Util_Media,N_PTDs,Ganho_LED,PFolga,PVE,D,Viabilidade_VE
0,Aveiro,Águeda,101,910.887701,244.320,0.268222,105715,0.477593,388,158.80800,50808.195148,5121.6,45845.403148,Viável
1,Aveiro,Albergaria-a-Velha,102,451.711801,28.020,0.062031,54540,0.469948,194,18.21300,26596.303834,2560.8,24053.716834,Viável
2,Aveiro,Anadia,103,657.071801,73.545,0.111928,55628,0.543009,223,47.80425,23387.762452,2943.6,20491.966702,Viável
3,Aveiro,Arouca,104,585.974400,115.720,0.197483,41884,0.527387,236,75.21800,18211.314133,3115.2,15171.332133,Viável
4,Aveiro,Aveiro,105,1055.192001,144.420,0.136866,197485,0.475475,509,93.87300,95298.999935,6718.8,88674.072935,Viável


In [2]:
# =========================================
# DROPDOWNS: Distrito -> Concelho
# =========================================

# Lista de distritos (ordenada)
lista_distritos = sorted(df_dashboard["Distrito"].dropna().unique())

# Dropdown de distrito
dropdown_distrito = widgets.Dropdown(
    options=lista_distritos,
    description="",
    layout=widgets.Layout(width="250px")
)

# Dropdown de concelho (atualizado dinamicamente)
dropdown_concelho = widgets.Dropdown(
    description="",
    layout=widgets.Layout(width="250px")
)

# Função para atualizar a lista de concelhos quando muda o distrito
def atualizar_concelhos(change):
    distrito = change["new"]

    lista_concelhos = sorted(
        df_dashboard.loc[df_dashboard["Distrito"] == distrito, "Concelho"]
        .dropna()
        .unique()
    )

    dropdown_concelho.options = lista_concelhos

    if lista_concelhos:
        dropdown_concelho.value = lista_concelhos[0]

# Ligar evento
dropdown_distrito.observe(atualizar_concelhos, names="value")

# Inicializar dropdowns
if lista_distritos:
    dropdown_distrito.value = lista_distritos[0]

In [3]:
def get_dados_concelho():
    # Obtém os dados do concelho selecionado nos dropdowns
    distrito = dropdown_distrito.value
    concelho = dropdown_concelho.value

    df_filtrado = df_dashboard.loc[
        (df_dashboard["Distrito"] == distrito) &
        (df_dashboard["Concelho"] == concelho)
        ]

    if df_filtrado.empty:
        return None

    return df_filtrado.iloc[0]

In [4]:
horas = list(range(24))

# Perfil horário normalizado (0–1) da iluminação pública ao longo do dia
# Valores elevados durante a noite e reduzidos durante o dia
perfil_iluminacao = np.array([
    0.75, 0.85, 0.95, 1.00, 1.00, 0.90, 0.60, 0.25,
    0.05, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.05,
    0.20, 0.50, 0.80, 0.95, 1.00, 1.00, 0.90, 0.80
])

assert len(perfil_iluminacao) == 24

In [5]:
def mostrar_metricas(row):
    estado = row["Viabilidade_VE"]
    emoji = "🟢" if estado == "Viável" else "🔴"
    cor = "green" if estado == "Viável" else "red"

    potencia_led = row["P_IP_TOTAL"] - row["Ganho_LED"]

    display(Markdown(f"""
## {row['Concelho']} ({row['Distrito']})

### ⚡ Energia e Infraestrutura
- **Potência total:** {row['P_IP_TOTAL']:.2f} kW  
- **Potência ineficiente:** {row['P_IP_Inef']:.2f} kW 
- **Estimativa Potência libertada (LED) pelas medidas de eficiência:** {row['Ganho_LED']:.2f} kW 
- **Potência após LED:** {potencia_led:.2f} kW  
- **Capacidade PTD:** {row['Cap_PTD']:.2f} kVA  
- **Utilização média:** {row['Util_Media']:.2%}  
- **Folga da rede:** {row['PFolga']:.2f}  

### 🚗 Mobilidade Elétrica
- **Carga VE estimada:** {row['PVE']:.2f}  

---

## Resultado Final

### **Saldo (D): {row['D']:.2f}**

### <span style="color:{cor}; font-size:18px;"><b>{emoji} {estado}</b></span>
"""))

In [6]:
def plot_perfil_horario(row):
    # Estimar consumo antes e depois da modernização LED
    consumo_antes = row["P_IP_TOTAL"] * perfil_iluminacao

    potencia_depois = max(row["P_IP_TOTAL"] - row["Ganho_LED"], 0)
    consumo_depois = potencia_depois * perfil_iluminacao

    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=horas,
        y=consumo_antes,
        mode="lines+markers",
        name="Antes (tecnologia convencional)"
    ))

    fig.add_trace(go.Scatter(
        x=horas,
        y=consumo_depois,
        mode="lines+markers",
        name="Depois (tecnologia LED)"
    ))

    fig.update_layout(
        title=f"Perfil horário de consumo — {row['Concelho']}",
        xaxis_title="Hora do dia",
        yaxis_title="Potência estimada (kW)",
        height=450
    )

    fig.show()

In [7]:
def plot_capacidade_viabilidade(row):
    capacidade_disponivel = row["PFolga"] + row["Ganho_LED"]
    carga_ve = row["PVE"]
    saldo = row["D"]

    categorias = ["Capacidade disponível", "Carga VE", "Saldo final"]
    valores = [capacidade_disponivel, carga_ve, saldo]

    cor_saldo = "#2E8B57" if saldo >= 0 else "#C0392B"

    fig = go.Figure(
        data=[
            go.Bar(
                x=categorias,
                y=valores,
                text=[f"{v:.2f}" for v in valores],
                textposition="outside",
                marker_color=["#B0BEC5", "#90A4AE", cor_saldo]
            )
        ]
    )

    fig.update_layout(
        title=f"Viabilidade da rede para carregadores VE — {row['Concelho']}",
        yaxis_title="Potência (kW)",
        template="plotly_white",
        height=470,
        showlegend=False,
        plot_bgcolor="white",
        paper_bgcolor="white",
        yaxis=dict(showgrid=True, gridcolor="#EAEAEA"),
        xaxis=dict(showgrid=False)
    )

    fig.show()

In [8]:
def plot_resumo_distrital():
    resumo = df_dashboard.groupby("Distrito", as_index=False).agg(
        Ganho_LED_Total=("Ganho_LED", "sum"),
        Capacidade_Total=("Cap_PTD", "sum"),
        Carga_VE_Total=("PVE", "sum"),
        Concelhos_Viaveis=("Viabilidade_VE", lambda x: (x == "Viável").sum())
    )

    fig = px.scatter(
        resumo,
        x="Capacidade_Total",
        y="Ganho_LED_Total",
        size="Concelhos_Viaveis",
        color="Distrito",
        hover_name="Distrito",
        hover_data={
            "Carga_VE_Total": True,
            "Concelhos_Viaveis": True
        },
        size_max=60,
        title="Resumo distrital: capacidade, ganho LED e concelhos viáveis",
    )

    fig.update_layout(
        xaxis_title="Capacidade total PTD (kVA)",
        yaxis_title="Ganho LED total (kW)",
        height=500
    )

    fig.show()

In [9]:
output_dashboard = widgets.Output()

In [10]:
def atualizar_dashboard(*args):
    with output_dashboard:
        output_dashboard.clear_output()

        row = get_dados_concelho()
        if row is None:
            print("Nenhum concelho selecionado.")
            return

        # KPIs
        display(Markdown(
            " #### Este painel resume os principais indicadores energéticos e de capacidade da rede para o concelho selecionado."
        ))
        mostrar_metricas(row)
        display(Markdown("---"))

        # Viabilidade
        display(Markdown(
            "#### Este gráfico compara a capacidade disponível da rede com a carga adicional dos veículos elétricos, permitindo avaliar a viabilidade da instalação de carregadores."
        ))
        plot_capacidade_viabilidade(row)
        display(Markdown("---"))

        # Perfil horário
        display(Markdown(
            "#### O perfil horário mostra a variação estimada do consumo de iluminação pública ao longo do dia, antes e depois da modernização para tecnologia LED."
        ))
        plot_perfil_horario(row)
        display(Markdown("---"))

        # Ranking
        display(Markdown(
            "#### Este gráfico identifica os concelhos com maior potencial de libertação de potência através da modernização LED."
        ))
        mostrar_top_ganho_led()
        display(Markdown("---"))

        # Resumo distrital
        display(Markdown(
            "####  gráfico seguinte relaciona a capacidade instalada e o ganho LED por distrito, destacando também o número de concelhos viáveis."
        ))
        plot_resumo_distrital()

In [11]:
dropdown_distrito.observe(atualizar_dashboard, names="value")
dropdown_concelho.observe(atualizar_dashboard, names="value")

display(widgets.HBox([dropdown_distrito, dropdown_concelho]))
display(output_dashboard)

atualizar_dashboard()

Output()